In [10]:
# importing useful libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pydytuesday

# Download files from the week, which you can then read in locally
# pydytuesday.get_date('2025-09-16')

# Option 2: Read directly from GitHub and assign to an object

"""
All Recipes columns:
'name', 'url', 'author', 'date_published', 'ingredients', 'calories',
'fat', 'carbs', 'protein', 'avg_rating', 'total_ratings', 'reviews',
'prep_time', 'cook_time', 'total_time', 'servings'
"""
all_recipes = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/all_recipes.csv')

"""
Cuisines columns:
'name', 'country', 'url', 'author', 'date_published', 'ingredients',
'calories', 'fat', 'carbs', 'protein', 'avg_rating', 'total_ratings',
'reviews', 'prep_time', 'cook_time', 'total_time', 'servings'
"""
cuisines = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/cuisines.csv')
print(cuisines.info())
print(cuisines.describe())


Trying to fetch README from: https://raw.githubusercontent.com/rfordatascience/tidytuesday/refs/heads/main/data/2025/2025-09-16/readme.md
Successfully fetched README from: https://raw.githubusercontent.com/rfordatascience/tidytuesday/refs/heads/main/data/2025/2025-09-16/readme.md
Successfully saved all_recipes.csv to d:\Education\DSA2101\DSA2101\all_recipes.csv
Successfully saved cuisines.csv to d:\Education\DSA2101\DSA2101\cuisines.csv
Successfully saved meta.yaml to d:\Education\DSA2101\DSA2101\meta.yaml
Successfully saved snack.png to d:\Education\DSA2101\DSA2101\snack.png
<class 'pandas.DataFrame'>
RangeIndex: 2218 entries, 0 to 2217
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            2218 non-null   str    
 1   country         2218 non-null   str    
 2   url             2218 non-null   str    
 3   author          2218 non-null   str    
 4   date_published  2218 non-null   str    
 5   i

In [16]:
"""
what are the factors that affect a recipe’s rating?

criterions to focus on: calories, country, total_time
"""

HADNLE_MISSING = "drop" # options: "drop", "impute"
print(len(cuisines))
if HADNLE_MISSING == "drop":
  # for fair comparison, we only consider cuisines with non-null country, calories, total_time, and avg_rating
  cuisines_handle_missing = cuisines.dropna(subset=['country', 'calories', 'total_time', 'avg_rating'])

  # ensure data types are correct: country -> categorical, calories and total_time -> numeric, avg_rating -> numeric
  cuisines_handle_missing['country'] = cuisines_handle_missing['country'].astype('category')
  cuisines_handle_missing['calories'] = cuisines_handle_missing['calories'].astype(float)
  cuisines_handle_missing['total_time'] = cuisines_handle_missing['total_time'].astype(float)
  cuisines_handle_missing['avg_rating'] = cuisines_handle_missing['avg_rating'].astype(float)

elif HADNLE_MISSING == "impute":
  # impute missing values with median for calories, total_time, and rating and unknown for country
  cuisines_handle_missing = cuisines.copy()
  cuisines_handle_missing['calories'] = cuisines_handle_missing['calories'].fillna(cuisines_handle_missing['calories'].median())
  cuisines_handle_missing['total_time'] = cuisines_handle_missing['total_time'].fillna(cuisines_handle_missing['total_time'].median())
  cuisines_handle_missing['country'] = cuisines_handle_missing['country'].fillna('unknown')
  cuisines_handle_missing['avg_rating'] = cuisines_handle_missing['avg_rating'].fillna(cuisines_handle_missing['avg_rating'].median())

print(len(cuisines_handle_missing))
"""
drop rows with 0 total_time, by looking at the urls of those recipes, 
we can see that they are missing the total_time data, 
not because they actually have 0 total_time
"""
cuisines_handle_missing = cuisines_handle_missing[cuisines_handle_missing['total_time'] > 0]
print(len(cuisines_handle_missing))



2218
2092
2043


In [14]:
# might consider removing outliers, in that case, use 
Q1 = cuisines_handle_missing[['calories', 'total_time']].quantile(0.25)
Q3 = cuisines_handle_missing[['calories', 'total_time']].quantile(0.75)
IQR = Q3 - Q1
print(Q1 - 1.5 * IQR)
print(Q3 + 1.5 * IQR)


cuisines_no_outliers = cuisines_handle_missing[~((cuisines_handle_missing[['calories', 'total_time']] < (Q1 - 1.5 * IQR)) | 
                                      (cuisines_handle_missing[['calories', 'total_time']] > (Q3 + 1.5 * IQR))).any(axis=1)]
print(len(cuisines_handle_missing))
print(len(cuisines_no_outliers))

REMOVE_OUTLIERS = False

if REMOVE_OUTLIERS:
  cuisines_cleaned = cuisines_no_outliers.copy()
else:
  cuisines_cleaned = cuisines_handle_missing.copy()

# group by each criterion and calculate the average rating for each group
cuisines_by_country = cuisines_cleaned.groupby('country')['avg_rating'].mean().reset_index().sort_values(by='avg_rating', ascending=False)
cuisines_by_calories = cuisines_cleaned[['calories', 'avg_rating']].copy()
cuisines_by_total_time = cuisines_cleaned[['total_time', 'avg_rating']].copy()

print(len(cuisines_by_country))
print(len(cuisines_by_calories))
print(len(cuisines_by_total_time))

calories     -237.5
total_time   -100.0
dtype: float64
calories      906.5
total_time    260.0
dtype: float64
2043
1765
49
2043
2043
